# Ep. 03 — LLM, contexto, limites e janela deslizante

Conteúdo realocado do Ep. 02, mais a janela deslizante: chamada simples, esquecimento, histórico, estouro e sliding window.


## 1. Chamada simples

Fazemos uma chamada a um LLM e mostramos a resposta.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("NVIDIA_API_KEY", "").strip()
MODEL = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-8b-instruct")

def ask(messages):
    """messages: string OU lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Defina NVIDIA_API_KEY e rode de novo.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

resposta = ask("Lembre deste segredo: a palavra-código é ORBIT-77.")
print(resposta)


## 2. Sem histórico

Agora pergunte ao modelo o que ele acabou de falar.


In [ ]:
# Nova chamada — sem passar o histórico da célula anterior
resposta2 = ask("O que você acabou de dizer?")
print(resposta2)


## 3. Montar o histórico

Para o modelo "lembrar", basta uma lista de mensagens `role` / `content`, com a última no final.


In [ ]:
historico = [
    {"role": "user", "content": "Lembre deste segredo: a palavra-código é ORBIT-77."},
    {"role": "assistant", "content": resposta},
    {"role": "user", "content": "Qual é a palavra-código?"},
]

resposta3 = ask(historico)
print(resposta3)


## 4. Estouro da janela

Se o histórico crescer demais, a janela de contexto estoura e o provedor devolve erro.


In [ ]:
# Cresce o histórico até a API falhar (ex.: 400 context length exceeded)
historico_grande = [
    {"role": "user", "content": "Lembre deste segredo: a palavra-código é ORBIT-77."},
    {"role": "assistant", "content": resposta},
]

bloco = "x" * 50_000  # texto grande só para encher o contexto

try:
    while True:
        historico_grande.append({"role": "user", "content": bloco})
        historico_grande.append({"role": "assistant", "content": "ok"})
        print(f"mensagens no histórico: {len(historico_grande)}")
        ask(historico_grande)
except Exception as e:
    print("Erro do provedor:")
    print(e)


## 5. Janela deslizante (sliding window)

Quando o histórico enche a janela de contexto, a mitigação mínima é **dropar as mensagens mais antigas** e manter só as recentes.

- O modelo **esquece o começo** da conversa.
- Mantém o **fio recente** e a chamada deixa de estourar.
- Isso não é sumarização: é só um recorte FIFO da lista de mensagens.

```text
[msg1][msg2][msg3][msg4][msg5][msg6]   ← estoura
              [msg3][msg4][msg5][msg6]   ← após sliding window (N=4)
```


In [ ]:
# Histórico longo com o segredo no COMEÇO e turnos curtos no final
historico_sw = [
    {"role": "user", "content": "Lembre deste segredo: a palavra-código é ORBIT-77."},
    {"role": "assistant", "content": resposta},
]
for i in range(1, 9):
    historico_sw.append({"role": "user", "content": f"Turno {i}: ok?"})
    historico_sw.append({"role": "assistant", "content": f"ok {i}"})

N = 6  # manter só as N mensagens mais recentes
janela = historico_sw[-N:]
print(f"Histórico original: {len(historico_sw)} msgs → janela: {len(janela)} msgs")
print("Primeira msg na janela:", janela[0]["content"][:80])

# Pergunta sobre o segredo (provavelmente esquecido se caiu fora da janela)
out_sw = ask(janela + [{"role": "user", "content": "Qual é a palavra-código?"}])
print(out_sw)
